# HumAID 04 — Anchor entity, JEV model and classifier
The three-stage pipeline, built up one component at a time so each one's effect is visible:

| Run | Confounder training data | Anchor entity + JEV heads | Contrastive loss |
|---|---|---|---|
| `baseline` | | | |
| `plus_confounders` | yes | | |
| `jev` | yes | yes | |
| `jev_contrastive` | yes | yes | yes |

Every run is scored on the official HumAID test set and on the confounder test split,
whose anchor entities never appear in training.

In [ ]:
import json
import os
import subprocess
import threading
import time

import pandas as pd
import torch

DEBUG = False  # True: 10% of the data and 1 epoch, to check the pipeline quickly
# One list per GPU, run top to bottom
QUEUES = [
    [("baseline", {"confounders": False, "jev": False, "contrastive": False}),
     ("plus_confounders", {"confounders": True, "jev": False, "contrastive": False})],
    [("jev", {"confounders": True, "jev": True, "contrastive": False}),
     ("jev_contrastive", {"confounders": True, "jev": True, "contrastive": True})],
]
OUT = "/kaggle/working"
T0 = time.time()


def log(msg):
    m, s = divmod(int(time.time() - T0), 60)
    print(f"[{m:02d}:{s:02d}] {msg}", flush=True)


n_gpu = torch.cuda.device_count()
assert n_gpu >= 1, "No GPU attached — enable it in the notebook settings"
if n_gpu == 1:
    QUEUES = [[job for q in QUEUES for job in q]]
log(f"GPUs: {n_gpu} x {torch.cuda.get_device_name(0)} | DEBUG={DEBUG}")

## 1. Project files used by the training script

In [ ]:
CONFOUNDER_CSV = 'id,anchor_entity,entity_type,split,reality,label,text,verified,corrected_reality,corrected_label,verifier,notes\nC01-1,bridge,transport,train,real_disaster,infrastructure_and_utility_damage,"Bridge near Aluva collapsed after the flood, vehicles stranded on both sides #KeralaFloods",,,,,\nC01-2,bridge,transport,train,real_disaster,infrastructure_and_utility_damage,"The old river bridge got washed away last night, our village is completely cut off now",,,,,\nC01-3,bridge,transport,train,real_disaster,caution_and_advice,"Cracks seen on the bridge after the quake, police asking everyone to stay off it",,,,,\nC01-4,bridge,transport,train,routine,not_humanitarian,"Bridge closed for repair work till Monday, please use the alternate route",,,,,\nC01-5,bridge,transport,train,routine,not_humanitarian,New bridge over the river finally opens next week after 3 years of construction,,,,,\nC01-6,bridge,transport,train,figurative,not_humanitarian,Burned that bridge with my ex long ago lol no going back,,,,,\nC01-7,bridge,transport,train,past_or_fiction,not_humanitarian,On this day in 1940 the Tacoma Narrows bridge collapsed. Still crazy to watch the footage,,,,,\nC01-8,bridge,transport,train,drill,not_humanitarian,"Mock drill on the bridge today 11am, rescue boats will be around, nothing to worry",,,,,\nC02-1,road,transport,train,real_disaster,infrastructure_and_utility_damage,"Main road to the hospital is under 4 feet of water, ambulances cant get through #flood",,,,,\nC02-2,road,transport,train,real_disaster,infrastructure_and_utility_damage,"Landslide has blocked the ghat road, several houses damaged, pls avoid",,,,,\nC02-3,road,transport,train,real_disaster,requests_or_urgent_needs,"Road to our colony washed out since morning, 40 families stuck with no food. please send help",,,,,\nC02-4,road,transport,train,routine,not_humanitarian,"Road blocked due to metro construction work near the station, expect delays",,,,,\nC02-5,road,transport,train,routine,not_humanitarian,"Road resurfacing on MG Road tonight 10pm-5am, traffic diverted",,,,,\nC02-6,road,transport,train,routine,not_humanitarian,"Road closed tomorrow morning for the city marathon, plan accordingly",,,,,\nC02-7,road,transport,train,figurative,not_humanitarian,Long road ahead for the team after that loss but we go again,,,,,\nC02-8,road,transport,train,past_or_fiction,not_humanitarian,"Watching a documentary on the 2013 floods, the roads just vanished. scary stuff",,,,,\nC03-1,highway,transport,train,real_disaster,infrastructure_and_utility_damage,"NH-44 caved in near Ramban after heavy rain, hundreds of trucks stranded",,,,,\nC03-2,highway,transport,train,real_disaster,infrastructure_and_utility_damage,"Wildfire has jumped the highway, both directions shut, people abandoning cars",,,,,\nC03-3,highway,transport,train,real_disaster,caution_and_advice,"Highway flooded near exit 12, do NOT try to drive through, turn around",,,,,\nC03-4,highway,transport,train,routine,not_humanitarian,"Highway widening work between km 40 and 52, single lane only this month",,,,,\nC03-5,highway,transport,train,routine,not_humanitarian,Toll plaza on the highway closed for 2 hrs for FASTag system upgrade,,,,,\nC03-6,highway,transport,train,figurative,not_humanitarian,My productivity is on a highway to hell this week,,,,,\nC03-7,highway,transport,train,past_or_fiction,not_humanitarian,Throwback to when the highway got snowed in back in 2019 and we played cricket on it,,,,,\nC03-8,highway,transport,train,drill,not_humanitarian,"Emergency landing drill on the highway airstrip today, traffic stopped 9 to 1",,,,,\nC04-1,flyover,transport,test,real_disaster,infrastructure_and_utility_damage,"Part of the flyover has come down after the earthquake, cars trapped underneath, need cranes urgently",,,,,\nC04-2,flyover,transport,test,real_disaster,infrastructure_and_utility_damage,"Flyover pillar is tilting after flood water eroded the base, traffic police have sealed it",,,,,\nC04-3,flyover,transport,test,real_disaster,injured_or_dead_people,Two bikers died when the flyover slab fell during the cyclone last night,,,,,\nC04-4,flyover,transport,test,routine,not_humanitarian,Flyover shut this weekend for expansion joint replacement,,,,,\nC04-5,flyover,transport,test,routine,not_humanitarian,"New flyover inauguration tomorrow by the CM, traffic diversions in place",,,,,\nC04-6,flyover,transport,test,routine,not_humanitarian,Demolition of the old flyover starts Friday to make way for the metro line,,,,,\nC04-7,flyover,transport,test,past_or_fiction,not_humanitarian,That scene in the movie where the flyover collapses looked so fake,,,,,\nC04-8,flyover,transport,test,drill,not_humanitarian,"Fire dept running a rescue drill under the flyover this morning, dont panic if you see smoke",,,,,\nC05-1,railway track,transport,train,real_disaster,infrastructure_and_utility_damage,"Railway tracks washed away near the station, all trains cancelled #floods",,,,,\nC05-2,railway track,transport,train,real_disaster,injured_or_dead_people,"Track buckled after the quake, train derailed, many passengers injured",,,,,\nC05-3,railway track,transport,train,real_disaster,requests_or_urgent_needs,"Tracks submerged for 2 km, passengers stuck on the train since 6 hrs without water",,,,,\nC05-4,railway track,transport,train,routine,not_humanitarian,"Track maintenance block on Sunday, 14 trains cancelled, check before you travel",,,,,\nC05-5,railway track,transport,train,routine,not_humanitarian,"Track doubling work finished, new line opens for trains from next month",,,,,\nC05-6,railway track,transport,train,figurative,not_humanitarian,This project has completely gone off track,,,,,\nC05-7,railway track,transport,train,figurative,not_humanitarian,Finally got my life back on track after a rough year,,,,,\nC05-8,railway track,transport,train,past_or_fiction,not_humanitarian,Reading about the 1981 Bihar train disaster where the train fell off the track into the river,,,,,\nC06-1,metro,transport,train,real_disaster,infrastructure_and_utility_damage,"Metro station flooded up to the platform, services suspended, people being pulled out",,,,,\nC06-2,metro,transport,train,real_disaster,infrastructure_and_utility_damage,"Metro pillar cracked after tremors, line shut and area cordoned off",,,,,\nC06-3,metro,transport,train,real_disaster,displaced_people_and_evacuations,"Water gushing into the metro tunnel, passengers evacuated on foot through the dark",,,,,\nC06-4,metro,transport,train,routine,not_humanitarian,Metro services on blue line suspended 6am-8am Sunday for signalling upgrade,,,,,\nC06-5,metro,transport,train,routine,not_humanitarian,Metro construction has dug up the whole road outside my office again,,,,,\nC06-6,metro,transport,train,routine,not_humanitarian,"Metro station closed for 10 mins due to a technical snag, running normal now",,,,,\nC06-7,metro,transport,train,figurative,not_humanitarian,"Morning metro was a disaster zone today, not even standing space",,,,,\nC06-8,metro,transport,train,drill,not_humanitarian,"Evacuation drill at the metro station at noon today, alarms will ring",,,,,\nC07-1,airport runway,transport,test,real_disaster,infrastructure_and_utility_damage,"Runway is under water, airport shut till further notice, all flights cancelled #ChennaiFloods",,,,,\nC07-2,airport runway,transport,test,real_disaster,infrastructure_and_utility_damage,"Cracks on the runway after the earthquake, relief flights being diverted",,,,,\nC07-3,airport runway,transport,test,real_disaster,infrastructure_and_utility_damage,"Debris all over the runway after the hurricane, no relief planes can land",,,,,\nC07-4,airport runway,transport,test,routine,not_humanitarian,Runway closed 6 hours daily for recarpeting work until March,,,,,\nC07-5,airport runway,transport,test,routine,not_humanitarian,"Second runway becomes operational from tomorrow, finally fewer delays",,,,,\nC07-6,airport runway,transport,test,figurative,not_humanitarian,"She owned that runway at fashion week, absolute fire",,,,,\nC07-7,airport runway,transport,test,past_or_fiction,not_humanitarian,Remember when the runway flooded in 2015? Airport has built new drains since then,,,,,\nC07-8,airport runway,transport,test,drill,not_humanitarian,"Full scale emergency drill on the runway today, fire tenders and ambulances are part of the exercise",,,,,\nC08-1,tunnel,transport,test,real_disaster,infrastructure_and_utility_damage,"Tunnel caved in after the landslide, workers trapped inside, rescue is on",,,,,\nC08-2,tunnel,transport,test,real_disaster,infrastructure_and_utility_damage,"Tunnel flooded completely, two cars submerged, divers called in",,,,,\nC08-3,tunnel,transport,test,real_disaster,caution_and_advice,"Smoke filling the tunnel after wildfire reached the entrance, traffic stopped both sides",,,,,\nC08-4,tunnel,transport,test,routine,not_humanitarian,Tunnel closed tonight for ventilation fan maintenance,,,,,\nC08-5,tunnel,transport,test,routine,not_humanitarian,"Breakthrough achieved in the new rail tunnel, engineers celebrating",,,,,\nC08-6,tunnel,transport,test,figurative,not_humanitarian,Finally seeing light at the end of the tunnel with this thesis,,,,,\nC08-7,tunnel,transport,test,past_or_fiction,not_humanitarian,The tunnel collapse in that Netflix series was so tense,,,,,\nC08-8,tunnel,transport,test,drill,not_humanitarian,"Mock rescue exercise inside the tunnel today, traffic will be held for 30 mins",,,,,\nC09-1,building,structure,train,real_disaster,infrastructure_and_utility_damage,"Building collapsed in the earthquake, people trapped inside, send help to Sector 9",,,,,\nC09-2,building,structure,train,real_disaster,infrastructure_and_utility_damage,"Roof of the apartment building ripped off by the hurricane, families on the street",,,,,\nC09-3,building,structure,train,real_disaster,displaced_people_and_evacuations,"Three-storey building tilted after flood water weakened the foundation, residents moved out",,,,,\nC09-4,building,structure,train,routine,not_humanitarian,Old building being demolished today to make way for a new mall,,,,,\nC09-5,building,structure,train,routine,not_humanitarian,"Scaffolding up on our building for repainting, dust everywhere",,,,,\nC09-6,building,structure,train,figurative,not_humanitarian,Team building exercise was a total trainwreck,,,,,\nC09-7,building,structure,train,past_or_fiction,not_humanitarian,Visited the memorial for the building collapse of 2013 today. Never forget,,,,,\nC09-8,building,structure,train,drill,not_humanitarian,"Fire drill in our office building at 3pm, everyone has to walk down 14 floors",,,,,\nC10-1,school building,structure,test,real_disaster,infrastructure_and_utility_damage,"School building roof caved in during the quake, thank god it was a Sunday",,,,,\nC10-2,school building,structure,test,real_disaster,infrastructure_and_utility_damage,"Our school is under water, books and benches all gone #floods",,,,,\nC10-3,school building,structure,test,real_disaster,infrastructure_and_utility_damage,"School wall fell on the road after the cyclone, power pole also down",,,,,\nC10-4,school building,structure,test,routine,not_humanitarian,"School building closed for summer renovation, new classrooms being added",,,,,\nC10-5,school building,structure,test,routine,not_humanitarian,"Old school block is being pulled down, new one will come up by next year",,,,,\nC10-6,school building,structure,test,figurative,not_humanitarian,Results day and the whole school is in ruins emotionally,,,,,\nC10-7,school building,structure,test,past_or_fiction,not_humanitarian,Our history teacher told us how the school was rebuilt after the 2001 earthquake,,,,,\nC10-8,school building,structure,test,drill,not_humanitarian,"Earthquake drill at school today, kids hid under desks and thought it was the best thing ever",,,,,\nC11-1,hospital,structure,train,real_disaster,infrastructure_and_utility_damage,"Hospital ground floor flooded, patients being shifted upstairs, generators failing",,,,,\nC11-2,hospital,structure,train,real_disaster,requests_or_urgent_needs,"Hospital running out of oxygen and beds after the quake, injured lying in corridors",,,,,\nC11-3,hospital,structure,train,real_disaster,requests_or_urgent_needs,"Roof of district hospital blown off by cyclone, need tarpaulin and volunteers",,,,,\nC11-4,hospital,structure,train,routine,not_humanitarian,"New wing of the hospital under construction, parking shifted to gate 3",,,,,\nC11-5,hospital,structure,train,routine,not_humanitarian,Hospital OPD closed tomorrow for the public holiday,,,,,\nC11-6,hospital,structure,train,figurative,not_humanitarian,My bank account needs to be hospitalized after this sale,,,,,\nC11-7,hospital,structure,train,past_or_fiction,not_humanitarian,The hospital scene in that disaster movie had me in tears,,,,,\nC11-8,hospital,structure,train,drill,not_humanitarian,"Mass casualty drill at the hospital today, the patients are all volunteers",,,,,\nC12-1,wall,structure,train,real_disaster,injured_or_dead_people,"Compound wall collapsed in heavy rain, 3 people buried under it",,,,,\nC12-2,wall,structure,train,real_disaster,infrastructure_and_utility_damage,"Sea wall breached by storm surge, water entering the fishing colony",,,,,\nC12-3,wall,structure,train,real_disaster,infrastructure_and_utility_damage,"Retaining wall gave way after landslide, road below is gone",,,,,\nC12-4,wall,structure,train,routine,not_humanitarian,"Wall being knocked down for road widening, shop owners got notice last month",,,,,\nC12-5,wall,structure,train,routine,not_humanitarian,"Painting the boundary wall this weekend, any colour suggestions?",,,,,\nC12-6,wall,structure,train,figurative,not_humanitarian,"Hit a wall with this assignment, brain not working",,,,,\nC12-7,wall,structure,train,figurative,not_humanitarian,Feels like talking to a wall with this customer care,,,,,\nC12-8,wall,structure,train,past_or_fiction,not_humanitarian,Berlin wall fell on this day in 1989,,,,,\nC13-1,dam,structure,train,real_disaster,caution_and_advice,"Dam gates opened without warning, villages downstream are flooding fast, move to higher ground",,,,,\nC13-2,dam,structure,train,real_disaster,displaced_people_and_evacuations,"Cracks reported in the dam after the earthquake, people downstream being evacuated",,,,,\nC13-3,dam,structure,train,real_disaster,infrastructure_and_utility_damage,"Dam has breached, water rushing into the town, houses washed away",,,,,\nC13-4,dam,structure,train,routine,not_humanitarian,"Dam gates to be opened tomorrow 10am for routine water release, river users please note",,,,,\nC13-5,dam,structure,train,routine,not_humanitarian,"Dam desilting work begins next week, boating suspended",,,,,\nC13-6,dam,structure,train,routine,not_humanitarian,"New dam project gets approval, construction to start in 2027",,,,,\nC13-7,dam,structure,train,past_or_fiction,not_humanitarian,Read about the 1979 Machchhu dam failure today. Thousands died. Heartbreaking history,,,,,\nC13-8,dam,structure,train,drill,not_humanitarian,"Siren test and mock drill at the dam on Saturday, do not panic",,,,,\nC14-1,embankment,structure,test,real_disaster,infrastructure_and_utility_damage,"River embankment breached at two places, water entering villages, people on rooftops",,,,,\nC14-2,embankment,structure,test,real_disaster,infrastructure_and_utility_damage,"Embankment washed away overnight, 500 acres of crops under water",,,,,\nC14-3,embankment,structure,test,real_disaster,requests_or_urgent_needs,"Embankment leaking near our village, need sandbags and volunteers right now",,,,,\nC14-4,embankment,structure,test,routine,not_humanitarian,"Embankment strengthening work before monsoon is going on, heavy vehicles not allowed",,,,,\nC14-5,embankment,structure,test,routine,not_humanitarian,Walking track on the river embankment closed for landscaping,,,,,\nC14-6,embankment,structure,test,routine,not_humanitarian,Govt sanctions funds for raising the embankment height by 2 metres,,,,,\nC14-7,embankment,structure,test,past_or_fiction,not_humanitarian,Grandpa still talks about the embankment breach of 1987,,,,,\nC14-8,embankment,structure,test,drill,not_humanitarian,Flood preparedness drill on the embankment today with NDRF boats,,,,,\nC15-1,power lines,utility,train,real_disaster,infrastructure_and_utility_damage,"Power lines down across the town after the cyclone, stay away from wet poles",,,,,\nC15-2,power lines,utility,train,real_disaster,injured_or_dead_people,"Live wire fell into flood water, two people electrocuted",,,,,\nC15-3,power lines,utility,train,real_disaster,infrastructure_and_utility_damage,"Wildfire took out the power lines, whole valley in the dark since last night",,,,,\nC15-4,power lines,utility,train,routine,not_humanitarian,Scheduled power cut tomorrow for maintenance of power lines,,,,,\nC15-5,power lines,utility,train,routine,not_humanitarian,"Underground cabling work in our street, overhead power lines being removed",,,,,\nC15-6,power lines,utility,train,routine,not_humanitarian,"Tree trimming near power lines this week, short outages expected",,,,,\nC15-7,power lines,utility,train,past_or_fiction,not_humanitarian,"Old photo of the 2012 grid failure, whole north India had no power for a day",,,,,\nC15-8,power lines,utility,train,drill,not_humanitarian,Utility crew training on dummy power lines at the depot today,,,,,\nC16-1,transformer,utility,train,real_disaster,infrastructure_and_utility_damage,"Transformer exploded in the flood water, sparks everywhere, whole area dark",,,,,\nC16-2,transformer,utility,train,real_disaster,infrastructure_and_utility_damage,"Transformer fell during the storm and crushed a parked car, wires on the road",,,,,\nC16-3,transformer,utility,train,real_disaster,infrastructure_and_utility_damage,"Transformer under water since 2 days, no power to pump out the flood water",,,,,\nC16-4,transformer,utility,train,routine,not_humanitarian,"Transformer being replaced in our lane, power off 10 to 4",,,,,\nC16-5,transformer,utility,train,routine,not_humanitarian,"New higher capacity transformer installed, hopefully no more voltage drops",,,,,\nC16-6,transformer,utility,train,routine,not_humanitarian,"Transformer oil being changed today, 2 hr shutdown",,,,,\nC16-7,transformer,utility,train,figurative,not_humanitarian,"Glow up of the year, total transformer",,,,,\nC16-8,transformer,utility,train,past_or_fiction,not_humanitarian,"Watched Transformers again, the city destruction scenes never get old",,,,,\nC17-1,substation,utility,test,real_disaster,infrastructure_and_utility_damage,"Substation flooded, power supply to half the city cut off, no timeline for restoration",,,,,\nC17-2,substation,utility,test,real_disaster,infrastructure_and_utility_damage,"Fire at the substation after lightning strike in the storm, hospitals on generators",,,,,\nC17-3,substation,utility,test,real_disaster,infrastructure_and_utility_damage,"Substation damaged in the quake, power out in 30 villages",,,,,\nC17-4,substation,utility,test,routine,not_humanitarian,Substation shutdown on Sunday 8am-2pm for annual maintenance,,,,,\nC17-5,substation,utility,test,routine,not_humanitarian,"New 220kV substation commissioned, should end low voltage complaints",,,,,\nC17-6,substation,utility,test,routine,not_humanitarian,"Substation upgrade work delayed again, contractor blames monsoon",,,,,\nC17-7,substation,utility,test,past_or_fiction,not_humanitarian,"The 2019 substation fire took a week to fix, hope they learned",,,,,\nC17-8,substation,utility,test,drill,not_humanitarian,Fire safety mock drill at the substation today,,,,,\nC18-1,electricity,utility,train,real_disaster,infrastructure_and_utility_damage,"No electricity since the hurricane hit 4 days ago, phones dead, food rotting",,,,,\nC18-2,electricity,utility,train,real_disaster,requests_or_urgent_needs,"Electricity gone in the whole district after the cyclone, hospital needs diesel for generators",,,,,\nC18-3,electricity,utility,train,real_disaster,requests_or_urgent_needs,"Power is out and water is rising, we are on the second floor with kids",,,,,\nC18-4,electricity,utility,train,routine,not_humanitarian,"Electricity will be off 9 to 5 tomorrow, board says annual maintenance",,,,,\nC18-5,electricity,utility,train,routine,not_humanitarian,Power cut again because someone didnt pay the society bill lol,,,,,\nC18-6,electricity,utility,train,routine,not_humanitarian,"Electricity tariff going up from next month, great",,,,,\nC18-7,electricity,utility,train,figurative,not_humanitarian,The electricity in that stadium tonight was unreal,,,,,\nC18-8,electricity,utility,train,figurative,not_humanitarian,Zero power left in me after this week,,,,,\nC19-1,water supply,utility,train,real_disaster,requests_or_urgent_needs,"No drinking water since the flood contaminated the supply, kids falling sick, need bottles",,,,,\nC19-2,water supply,utility,train,real_disaster,infrastructure_and_utility_damage,"Water supply cut after the quake broke the mains, people queuing at tankers",,,,,\nC19-3,water supply,utility,train,real_disaster,caution_and_advice,"Water treatment plant flooded, city told to boil water before drinking",,,,,\nC19-4,water supply,utility,train,routine,not_humanitarian,"Water supply shut tomorrow for tank cleaning, store water tonight",,,,,\nC19-5,water supply,utility,train,routine,not_humanitarian,Low water pressure this week due to pump replacement,,,,,\nC19-6,water supply,utility,train,routine,not_humanitarian,"Water bill doubled this month, how",,,,,\nC19-7,water supply,utility,train,figurative,not_humanitarian,My motivation supply has been cut off,,,,,\nC19-8,water supply,utility,train,past_or_fiction,not_humanitarian,During the 2019 drought the city almost ran out of water. Day zero they called it,,,,,\nC20-1,water pipeline,utility,train,real_disaster,infrastructure_and_utility_damage,"Main pipeline burst in the earthquake, road flooded, no water in taps",,,,,\nC20-2,water pipeline,utility,train,real_disaster,infrastructure_and_utility_damage,"Pipeline washed away in the landslide, 12 villages without water",,,,,\nC20-3,water pipeline,utility,train,real_disaster,infrastructure_and_utility_damage,"Sewage pipeline broken by flood, dirty water entering homes",,,,,\nC20-4,water pipeline,utility,train,routine,not_humanitarian,"Pipeline laying work on our road, dug up for a month now",,,,,\nC20-5,water pipeline,utility,train,routine,not_humanitarian,"Pipeline leak near the market being fixed, supply back by evening",,,,,\nC20-6,water pipeline,utility,train,routine,not_humanitarian,New pipeline will bring river water to the city by 2028,,,,,\nC20-7,water pipeline,utility,train,figurative,not_humanitarian,"So many projects in the pipeline, zero time",,,,,\nC20-8,water pipeline,utility,train,past_or_fiction,not_humanitarian,Remember the big pipeline burst last summer? Road is still patchy,,,,,\nC21-1,gas pipeline,utility,test,real_disaster,displaced_people_and_evacuations,"Gas pipeline ruptured in the quake, fire spreading, area being evacuated",,,,,\nC21-2,gas pipeline,utility,test,real_disaster,caution_and_advice,"Strong gas smell after the flood shifted the pipeline, police clearing the street",,,,,\nC21-3,gas pipeline,utility,test,real_disaster,infrastructure_and_utility_damage,"Gas line exploded after the hurricane damaged it, several houses burnt",,,,,\nC21-4,gas pipeline,utility,test,routine,not_humanitarian,"Piped gas connection work in our society this week, digging near gate 2",,,,,\nC21-5,gas pipeline,utility,test,routine,not_humanitarian,"Gas pipeline pressure testing tonight, you may hear a hissing sound",,,,,\nC21-6,gas pipeline,utility,test,routine,not_humanitarian,Gas supply off for 3 hrs tomorrow for meter replacement,,,,,\nC21-7,gas pipeline,utility,test,past_or_fiction,not_humanitarian,The pipeline explosion scene in that film was insane,,,,,\nC21-8,gas pipeline,utility,test,drill,not_humanitarian,"Gas leak mock drill at the plant today, sirens are part of it",,,,,\nC22-1,mobile tower,utility,test,real_disaster,infrastructure_and_utility_damage,"Mobile tower fell in the cyclone, no network anywhere, cant reach family",,,,,\nC22-2,mobile tower,utility,test,real_disaster,infrastructure_and_utility_damage,"All towers down after the hurricane, we have no way to call for help",,,,,\nC22-3,mobile tower,utility,test,real_disaster,injured_or_dead_people,"Tower collapsed on a house during the storm, one person injured",,,,,\nC22-4,mobile tower,utility,test,routine,not_humanitarian,"New mobile tower coming up near the park, residents protesting",,,,,\nC22-5,mobile tower,utility,test,routine,not_humanitarian,"Tower being upgraded to 5G, network may drop for few hours",,,,,\nC22-6,mobile tower,utility,test,routine,not_humanitarian,"Old mobile tower dismantled today, finally",,,,,\nC22-7,mobile tower,utility,test,figurative,not_humanitarian,He is a tower of strength for this team,,,,,\nC22-8,mobile tower,utility,test,past_or_fiction,not_humanitarian,After the 2018 floods they put towers on wheels for emergencies,,,,,\nC23-1,internet,utility,train,real_disaster,requests_or_urgent_needs,"No network since the earthquake, please someone check on my parents in Bhuj",,,,,\nC23-2,internet,utility,train,real_disaster,infrastructure_and_utility_damage,"Internet and phones down across the island after the hurricane, cut off from the world",,,,,\nC23-3,internet,utility,train,real_disaster,infrastructure_and_utility_damage,"Network down in flood hit areas, rescue teams cant coordinate",,,,,\nC23-4,internet,utility,train,routine,not_humanitarian,Internet down for 2 hrs tonight for fibre maintenance says the provider,,,,,\nC23-5,internet,utility,train,routine,not_humanitarian,"Wifi is down again at the hostel, how do I submit the assignment",,,,,\nC23-6,internet,utility,train,routine,not_humanitarian,Network shutdown in the exam centre area to stop cheating,,,,,\nC23-7,internet,utility,train,figurative,not_humanitarian,That meme broke the internet,,,,,\nC23-8,internet,utility,train,figurative,not_humanitarian,My brain has no network before coffee,,,,,\nC24-1,fire,hazard,train,real_disaster,displaced_people_and_evacuations,"Fire has reached the edge of town, we can see flames from our street, leaving now",,,,,\nC24-2,fire,hazard,train,real_disaster,infrastructure_and_utility_damage,"Houses on fire after gas lines broke in the quake, fire engines cant get through",,,,,\nC24-3,fire,hazard,train,real_disaster,infrastructure_and_utility_damage,"Wildfire jumped the river, 200 homes gone #CampFire",,,,,\nC24-4,fire,hazard,train,routine,not_humanitarian,"Controlled burning of crop stubble in the fields today, smoke is expected",,,,,\nC24-5,fire,hazard,train,figurative,not_humanitarian,"That concert was fire, the crowd was flooding in",,,,,\nC24-6,fire,hazard,train,figurative,not_humanitarian,This new album is straight fire,,,,,\nC24-7,fire,hazard,train,past_or_fiction,not_humanitarian,Great Fire of London started on this day in 1666,,,,,\nC24-8,fire,hazard,train,drill,not_humanitarian,"Fire drill at 11, please use the stairs",,,,,\nC25-1,smoke,hazard,train,real_disaster,requests_or_urgent_needs,"Thick smoke from the wildfire, cant see across the street, kids coughing, masks needed",,,,,\nC25-2,smoke,hazard,train,real_disaster,infrastructure_and_utility_damage,"Smoke pouring out of the collapsed factory after the quake, people still inside",,,,,\nC25-3,smoke,hazard,train,real_disaster,caution_and_advice,"Air is unbreathable, ash falling like snow, authorities say stay indoors",,,,,\nC25-4,smoke,hazard,train,routine,not_humanitarian,"Smoke near the highway is from road tarring work, not a fire",,,,,\nC25-5,smoke,hazard,train,routine,not_humanitarian,"Fogging for mosquitoes in our colony today, so much smoke",,,,,\nC25-6,smoke,hazard,train,figurative,not_humanitarian,We absolutely smoked them in the finals,,,,,\nC25-7,smoke,hazard,train,past_or_fiction,not_humanitarian,The smoke scenes in the Chernobyl series are haunting,,,,,\nC25-8,smoke,hazard,train,drill,not_humanitarian,Smoke you see at the mall is from a fire safety demo,,,,,\nC26-1,siren,hazard,train,real_disaster,caution_and_advice,"Tsunami sirens going off, everyone running to higher ground, please move now",,,,,\nC26-2,siren,hazard,train,real_disaster,injured_or_dead_people,"Sirens all night, ambulances bringing injured from the collapsed market",,,,,\nC26-3,siren,hazard,train,real_disaster,caution_and_advice,"Tornado siren just went off, get to the basement now",,,,,\nC26-4,siren,hazard,train,routine,not_humanitarian,"Siren you heard at 10 was the monthly test, all fine",,,,,\nC26-5,siren,hazard,train,routine,not_humanitarian,Factory siren for shift change scares me every single time,,,,,\nC26-6,siren,hazard,train,figurative,not_humanitarian,"Her voice is a siren song, cant stop listening",,,,,\nC26-7,siren,hazard,train,past_or_fiction,not_humanitarian,Old air raid sirens from the war are now in the museum,,,,,\nC26-8,siren,hazard,train,drill,not_humanitarian,"Civil defence siren drill tomorrow 4pm, no need to panic",,,,,\nC27-1,evacuation,hazard,train,real_disaster,displaced_people_and_evacuations,"Mandatory evacuation for zone A, storm surge expected by evening, leave now",,,,,\nC27-2,evacuation,hazard,train,real_disaster,displaced_people_and_evacuations,"We have been evacuated to the school, left everything behind, dont know if house is standing",,,,,\nC27-3,evacuation,hazard,train,real_disaster,displaced_people_and_evacuations,"Evacuating my grandparents from the fire zone, roads jammed",,,,,\nC27-4,evacuation,hazard,train,routine,not_humanitarian,"Evacuation of the old market for redevelopment starts next week, shopkeepers get new stalls",,,,,\nC27-5,evacuation,hazard,train,routine,not_humanitarian,"Building evacuated for 10 mins because someone burnt toast, false alarm",,,,,\nC27-6,evacuation,hazard,train,figurative,not_humanitarian,Mass evacuation from the group chat after that message,,,,,\nC27-7,evacuation,hazard,train,past_or_fiction,not_humanitarian,Dunkirk evacuation is still the most incredible story,,,,,\nC27-8,evacuation,hazard,train,drill,not_humanitarian,"Evacuation drill at the office today, took us 12 mins to get out",,,,,\nC28-1,shelter,hazard,test,real_disaster,requests_or_urgent_needs,"Shelter at the school is full, 300 people and only 2 toilets, need blankets and food",,,,,\nC28-2,shelter,hazard,test,real_disaster,requests_or_urgent_needs,"Looking for a shelter that takes pets, we had to leave because of the fire",,,,,\nC28-3,shelter,hazard,test,real_disaster,rescue_volunteering_or_donation_effort,"Relief shelter opened at the town hall for flood affected families, volunteers welcome",,,,,\nC28-4,shelter,hazard,test,routine,not_humanitarian,New bus shelter installed at our stop finally,,,,,\nC28-5,shelter,hazard,test,routine,not_humanitarian,"Animal shelter adoption drive this Sunday, come meet the puppies",,,,,\nC28-6,shelter,hazard,test,routine,not_humanitarian,Night shelter for the homeless being renovated before winter,,,,,\nC28-7,shelter,hazard,test,figurative,not_humanitarian,Taking shelter from my responsibilities under this blanket,,,,,\nC28-8,shelter,hazard,test,drill,not_humanitarian,Cyclone shelter mock drill held in coastal villages today,,,,,\nC29-1,ambulance,hazard,test,real_disaster,requests_or_urgent_needs,"Ambulances cant reach us, road is flooded, my father needs dialysis, please help",,,,,\nC29-2,ambulance,hazard,test,real_disaster,injured_or_dead_people,"Ambulances lined up outside the collapsed building, bringing out injured one by one",,,,,\nC29-3,ambulance,hazard,test,real_disaster,requests_or_urgent_needs,"Need ambulance in ward 12, woman in labour and water is waist deep",,,,,\nC29-4,ambulance,hazard,test,routine,not_humanitarian,New fleet of ambulances flagged off by the health minister today,,,,,\nC29-5,ambulance,hazard,test,routine,not_humanitarian,"Ambulance stuck in regular evening traffic again, people please give way",,,,,\nC29-6,ambulance,hazard,test,figurative,not_humanitarian,"Someone call an ambulance for that bowler, he got destroyed",,,,,\nC29-7,ambulance,hazard,test,past_or_fiction,not_humanitarian,The ambulance chase in that movie was the best part,,,,,\nC29-8,ambulance,hazard,test,drill,not_humanitarian,"Ambulances at the stadium are for the emergency drill, relax",,,,,\nC30-1,rescue team,hazard,train,real_disaster,rescue_volunteering_or_donation_effort,"Rescue team reached our village by boat, 60 people brought to safety #KeralaFloods",,,,,\nC30-2,rescue team,hazard,train,real_disaster,rescue_volunteering_or_donation_effort,"NDRF pulling survivors from the rubble, 4 rescued so far",,,,,\nC30-3,rescue team,hazard,train,real_disaster,requests_or_urgent_needs,"Still waiting for rescue, 3rd day on the roof, water not going down",,,,,\nC30-4,rescue team,hazard,train,routine,not_humanitarian,"Rescued a kitten from the drain today, she is home now",,,,,\nC30-5,rescue team,hazard,train,routine,not_humanitarian,"Lifeguard rescue training batch starts next month, registrations open",,,,,\nC30-6,rescue team,hazard,train,figurative,not_humanitarian,Coffee to the rescue once again,,,,,\nC30-7,rescue team,hazard,train,past_or_fiction,not_humanitarian,The cave rescue in Thailand in 2018 was a miracle,,,,,\nC30-8,rescue team,hazard,train,drill,not_humanitarian,"NDRF team doing a flood rescue demo at the lake today, come watch",,,,,\n'
ANCHOR_ENTITIES_PY = '"""Stage 1 of the pipeline: find the anchor entity of a tweet with a word list.\n\nThe anchor entity is the thing the tweet is about (a bridge, the power supply, a shelter).\n`tag(text)` wraps the first mention in [E] ... [/E] markers so the model can read the\nentity together with the event around it.\n\nRun this file directly to print coverage on the local data:\n    python src/anchor_entities.py\n"""\nimport re\n\nTERMS = [\n    # utilities\n    r"power lines?", r"power poles?", r"power grid", r"power supply", r"power", r"electricity", r"electric poles?",\n    r"wires?", r"transformers?", r"substations?", r"water supply", r"drinking water", r"water mains?", r"water",\n    r"gas pipelines?", r"gas lines?", r"pipelines?", r"gas", r"mobile towers?", r"cell towers?", r"towers?",\n    r"network", r"internet", r"wifi", r"phone lines?", r"cell service",\n    # transport\n    r"bridges?", r"roads?", r"highways?", r"freeways?", r"interstate", r"nh-?\\d+", r"streets?", r"flyovers?",\n    r"overpass(?:es)?", r"railway tracks?", r"rail tracks?", r"tracks?", r"railways?", r"trains?", r"metro",\n    r"subway", r"airports?", r"runways?", r"tunnels?", r"ports?", r"harbou?rs?", r"ferr(?:y|ies)",\n    # structures\n    r"school buildings?", r"buildings?", r"houses?", r"homes?", r"schools?", r"hospitals?", r"walls?", r"roofs?",\n    r"dams?", r"levees?", r"embankments?", r"church(?:es)?", r"mosques?", r"temples?", r"hotels?",\n    r"apartments?", r"malls?", r"factor(?:y|ies)", r"markets?", r"shops?",\n    # hazards and response\n    r"wildfires?", r"fires?", r"smoke", r"sirens?", r"evacuat\\w+", r"shelters?", r"ambulances?", r"rescue\\w*",\n    r"ndrf",\n]\n# Longest alternative first, so "power lines" wins over "power" at the same position\nANCHOR_RE = re.compile(r"\\b(?:" + "|".join(sorted(TERMS, key=len, reverse=True)) + r")\\b", re.IGNORECASE)\n\n\ndef find(text):\n    """Return (start, end) of the first anchor entity mention, or None."""\n    m = ANCHOR_RE.search(text)\n    return (m.start(), m.end()) if m else None\n\n\ndef tag(text):\n    """Wrap the first anchor entity mention in [E] ... [/E]; unchanged if there is none."""\n    span = find(text)\n    if span is None:\n        return text\n    s, e = span\n    return f"{text[:s]}[E] {text[s:e]} [/E]{text[e:]}"\n\n\nif __name__ == "__main__":\n    import os\n    from collections import Counter\n\n    import pandas as pd\n\n    root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\n    conf = pd.read_csv(os.path.join(root, "confounder_set", "confounder_set_v1_draft.csv"))\n    found = conf["text"].map(lambda t: find(t) is not None)\n    print(f"confounder set: anchor found in {found.mean():.1%} of {len(conf)} sentences")\n    for _, r in conf[~found].iterrows():\n        print("   no anchor:", r["id"], "|", r["text"])\n\n    path = os.path.join(root, "data", "humaid.csv")\n    if os.path.exists(path):\n        hum = pd.read_csv(path)\n        mentions = hum["text"].map(lambda t: (lambda s: t[s[0]:s[1]].lower() if s else None)(find(t)))\n        print(f"\\nHumAID: anchor found in {mentions.notna().mean():.1%} of {len(hum)} tweets")\n        print(hum.assign(found=mentions.notna()).groupby("label")["found"].mean().round(3).to_string())\n        print("\\nmost common anchors:", Counter(mentions.dropna()).most_common(25))\n'
for name, content in [("confounder.csv", CONFOUNDER_CSV), ("anchor_entities.py", ANCHOR_ENTITIES_PY)]:
    with open(name, "w", encoding="utf-8") as f:
        f.write(content)
conf = pd.read_csv("confounder.csv")
log(f"confounder set: {len(conf)} sentences, {conf['anchor_entity'].nunique()} anchor entities, "
    f"verified: {conf['verified'].notna().sum()}")

## 2. Training script (one run per call)

In [ ]:
WORKER = r'''
import glob, json, os, random, re, sys, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from anchor_entities import find, tag

NAME, CFG = sys.argv[1], json.loads(sys.argv[2])
DEBUG = os.environ.get("DEBUG") == "1"
CHECKPOINT, LR = "answerdotai/ModernBERT-base", 5e-5
EPOCHS, HUM_BATCH, CONF_PER_BATCH, MAX_LEN, SEED, LOG_EVERY = (1 if DEBUG else 3), 28, 4, 96, 42, 100
W_REAL, W_CON, TEMP = 0.5, 0.3, 0.1   # loss weights for the reality head and contrastive term
OUT = "/kaggle/working"
T0 = float(os.environ["T0"])
device = torch.device("cuda")
rng = random.Random(SEED)
torch.manual_seed(SEED)


def log(msg):
    m, s = divmod(int(time.time() - T0), 60)
    print(f"[{m:02d}:{s:02d}] {NAME} | {msg}", flush=True)


def clean(text):
    text = re.sub(r"https?://\S+|www\.\S+", "http", text)
    text = re.sub(r"@\w+", "@user", text)
    text = re.sub(r"^RT\s+", "", text.strip())
    text = text.replace("&amp;", "&")
    return re.sub(r"\s+", " ", text).strip()


# ---------- data
hum = pd.read_csv(glob.glob("/kaggle/input/**/humaid.csv", recursive=True)[0])
conf = pd.read_csv(f"{OUT}/confounder.csv")
labels = sorted(hum["label"].unique())
label2id = {l: i for i, l in enumerate(labels)}
NH = label2id["not_humanitarian"]
for d in (hum, conf):
    d["clean"] = d["text"].map(clean)
    d["y"] = d["label"].map(label2id)
    d["has_anchor"] = d["clean"].map(lambda t: find(t) is not None)
    # Stage 1: mark the anchor entity so the model can read it together with its event
    d["model_text"] = d["clean"].map(tag) if CFG["jev"] else d["clean"]
hum["real"] = (hum["y"] != NH).astype(int)
conf["real"] = (conf["reality"] == "real_disaster").astype(int)

train, dev, test = (hum[hum["split"] == s].reset_index(drop=True) for s in ["train", "dev", "test"])
conf_train = conf[conf["split"] == "train"].reset_index(drop=True)
conf_test = conf[conf["split"] == "test"].reset_index(drop=True)
assert not set(conf_train["anchor_entity"]) & set(conf_test["anchor_entity"])
if DEBUG:
    train = train.sample(frac=0.1, random_state=SEED).reset_index(drop=True)

tok = AutoTokenizer.from_pretrained(CHECKPOINT)
if CFG["jev"]:
    tok.add_special_tokens({"additional_special_tokens": ["[E]", "[/E]"]})
    E_ID = tok.convert_tokens_to_ids("[E]")


def encode(d):
    enc = tok(list(d["model_text"]), truncation=True, max_length=MAX_LEN)
    return [{"input_ids": enc["input_ids"][i], "attention_mask": enc["attention_mask"][i],
             "y": int(d["y"].iloc[i]), "real": int(d["real"].iloc[i])} for i in range(len(d))]


def collate(items):
    batch = tok.pad({"input_ids": [it["input_ids"] for it in items],
                     "attention_mask": [it["attention_mask"] for it in items]}, return_tensors="pt")
    batch = {k: v.to(device) for k, v in batch.items()}
    y = torch.tensor([it["y"] for it in items], device=device)
    real = torch.tensor([it["real"] for it in items], device=device)
    return batch, y, real


train_items, dev_items, test_items = encode(train), encode(dev), encode(test)
conf_test_items = encode(conf_test)
conf_groups = []   # per anchor entity: (real sentences, look-alike sentences)
if CFG["confounders"]:
    conf_train_items = encode(conf_train)
    for _, idx in conf_train.groupby("anchor_entity").indices.items():
        conf_groups.append(([conf_train_items[i] for i in idx if conf_train["real"].iloc[i] == 1],
                            [conf_train_items[i] for i in idx if conf_train["real"].iloc[i] == 0]))

counts = train["y"].value_counts().sort_index().values
cat_weights = torch.tensor(np.sqrt(counts.max() / counts), dtype=torch.float, device=device)
n_real = int(train["real"].sum())
real_weights = torch.tensor([np.sqrt(n_real / (len(train) - n_real)), 1.0], dtype=torch.float, device=device)


# ---------- model
class TriageModel(nn.Module):
    """Plain: sentence vector -> category. JEV: [sentence ; anchor entity] -> reality and category."""

    def __init__(self):
        super().__init__()
        self.enc = AutoModel.from_pretrained(CHECKPOINT).float()
        if CFG["jev"]:
            self.enc.resize_token_embeddings(len(tok))
        h = self.enc.config.hidden_size
        d = 2 * h if CFG["jev"] else h
        self.cat_head = nn.Sequential(nn.Linear(d, h), nn.GELU(), nn.Dropout(0.1), nn.Linear(h, len(labels)))
        if CFG["jev"]:
            self.real_head = nn.Sequential(nn.Linear(d, h), nn.GELU(), nn.Dropout(0.1), nn.Linear(h, 2))
            self.proj = nn.Linear(d, 256)

    def forward(self, input_ids, attention_mask):
        hid = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(hid.dtype)
        sent = (hid * mask).sum(1) / mask.sum(1)
        if not CFG["jev"]:
            return self.cat_head(sent), None, None
        # Position of the [E] marker; 0 means the tweet has no anchor entity, so fall back to the sentence
        ent_pos = (input_ids == E_ID).int().argmax(1)
        ent = hid[torch.arange(hid.size(0), device=hid.device), ent_pos]
        ent = torch.where((ent_pos > 0).unsqueeze(-1), ent, sent)
        joint = torch.cat([sent, ent], dim=-1)
        return self.cat_head(joint), self.real_head(joint), self.proj(joint)


def supcon(z, y):
    """Supervised contrastive loss: same reality pulled together, different reality pushed apart."""
    z = F.normalize(z, dim=1)
    sim = z @ z.T / TEMP
    eye = torch.eye(len(z), dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(eye, -1e9)
    pos = (y[:, None] == y[None, :]) & ~eye
    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    valid = pos.sum(1) > 0
    if not valid.any():
        return z.sum() * 0
    return (-(log_prob * pos).sum(1)[valid] / pos.sum(1)[valid]).mean()


@torch.no_grad()
def predict(items):
    """Returns final prediction (stage 3, gated by stage 2), ungated prediction, and P(real)."""
    model.eval()
    cat, real = [], []
    for i in range(0, len(items), 128):
        batch, _, _ = collate(items[i:i + 128])
        with torch.autocast("cuda", dtype=torch.float16):
            c, r, _ = model(**batch)
        cat.append(c.float().softmax(-1).cpu())
        if r is not None:
            real.append(r.float().softmax(-1).cpu())
    cat = torch.cat(cat).numpy()
    ungated = cat.argmax(-1)
    if not real:
        return ungated, ungated, 1 - cat[:, NH]
    p_real = torch.cat(real).numpy()[:, 1]
    return np.where(p_real < 0.5, NH, ungated), ungated, p_real


def scores(y_true, y_pred):
    return {"accuracy": round(accuracy_score(y_true, y_pred), 4),
            "macro_f1": round(f1_score(y_true, y_pred, average="macro"), 4),
            "weighted_f1": round(f1_score(y_true, y_pred, average="weighted"), 4)}


def confounder_scores(pred):
    is_real, said_nh = conf_test["real"].values == 1, pred == NH
    out = {"real_kept": float(np.mean(~said_nh[is_real])),
           "real_exact_category": float(np.mean(pred[is_real] == conf_test["y"].values[is_real])),
           "lookalike_rejected": float(np.mean(said_nh[~is_real]))}
    out["balanced"] = (out["real_kept"] + out["lookalike_rejected"]) / 2
    for t in ["routine", "figurative", "past_or_fiction", "drill"]:
        out[f"rejected_{t}"] = float(np.mean(said_nh[(conf_test["reality"] == t).values]))
    return {k: round(v, 3) for k, v in out.items()}


# ---------- training
log(f"config {CFG} | train={len(train)} | anchor found: HumAID train {train['has_anchor'].mean():.1%}, "
    f"confounder test {conf_test['has_anchor'].mean():.1%}")
model = TriageModel().to(device)
loader = DataLoader(train_items, batch_size=HUM_BATCH, shuffle=True, collate_fn=list)
total = EPOCHS * len(loader)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = get_linear_schedule_with_warmup(opt, int(0.06 * total), total)
scaler = torch.amp.GradScaler("cuda")

best_f1, best_state, history, step, start = -1, None, [], 0, time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    run = {"cat": 0.0, "real": 0.0, "con": 0.0}
    for items in loader:
        if conf_groups:
            # One anchor entity per batch: 2 real reports and 2 look-alikes, i.e. hard negatives for each other
            reals, fakes = rng.choice(conf_groups)
            items = items + rng.sample(reals, CONF_PER_BATCH // 2) + rng.sample(fakes, CONF_PER_BATCH // 2)
        batch, y, real = collate(items)
        with torch.autocast("cuda", dtype=torch.float16):
            cat_logits, real_logits, z = model(**batch)
        loss = l_cat = F.cross_entropy(cat_logits.float(), y, weight=cat_weights)
        run["cat"] += l_cat.item()
        if CFG["jev"]:
            l_real = F.cross_entropy(real_logits.float(), real, weight=real_weights)
            loss = loss + W_REAL * l_real
            run["real"] += l_real.item()
        if CFG["contrastive"]:
            l_con = supcon(z.float(), real)
            loss = loss + W_CON * l_con
            run["con"] += l_con.item()
        if not torch.isfinite(loss):
            log("loss is not finite — stopping this run")
            sys.exit(1)
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        step += 1
        if step % LOG_EVERY == 0:
            rate = (time.time() - start) / step
            parts = " ".join(f"{k} {v / LOG_EVERY:.3f}" for k, v in run.items() if v)
            log(f"epoch {epoch}/{EPOCHS} step {step}/{total} loss {parts} | {1 / rate:.1f} it/s | "
                f"ETA {rate * (total - step) / 60:.1f} min")
            run = dict.fromkeys(run, 0.0)
    dev_scores = scores(dev["y"], predict(dev_items)[0])
    conf_now = confounder_scores(predict(conf_test_items)[0])
    history.append({"epoch": epoch, **dev_scores, "confounder_balanced": conf_now["balanced"]})
    log(f"epoch {epoch} DEV {dev_scores} | CONFOUNDER real_kept {conf_now['real_kept']} "
        f"lookalike_rejected {conf_now['lookalike_rejected']}")
    if dev_scores["macro_f1"] > best_f1:   # selection uses HumAID dev only, never the confounder test
        best_f1 = dev_scores["macro_f1"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

# ---------- final evaluation
model.load_state_dict(best_state)
test_pred, test_ungated, _ = predict(test_items)
conf_pred, _, conf_p_real = predict(conf_test_items)
report = classification_report(test["y"], test_pred, target_names=labels, digits=3, output_dict=True)
result = {"config": CFG, "checkpoint": CHECKPOINT, "lr": LR, "dev_history": history,
          "humaid_test": {**scores(test["y"], test_pred),
                          "macro_f1_ungated": round(f1_score(test["y"], test_ungated, average="macro"), 4)},
          "per_class_f1": {l: round(report[l]["f1-score"], 3) for l in labels},
          "confounder_test": {"n_real": int(conf_test["real"].sum()),
                              "n_lookalike": int((conf_test["real"] == 0).sum()), **confounder_scores(conf_pred)},
          "anchor_found": {"humaid_train": round(float(train["has_anchor"].mean()), 3),
                           "confounder_test": round(float(conf_test["has_anchor"].mean()), 3)},
          "train_minutes": round((time.time() - start) / 60, 1)}
log(f"HUMAID TEST {result['humaid_test']}")
log(f"CONFOUNDER TEST {result['confounder_test']}")

pd.DataFrame({"tweet_id": test["tweet_id"], "event": test["event"], "label": test["label"],
              "pred": [labels[i] for i in test_pred]}).to_csv(f"{OUT}/{NAME}_humaid_test_predictions.csv", index=False)
pd.DataFrame({"id": conf_test["id"], "anchor_entity": conf_test["anchor_entity"], "reality": conf_test["reality"],
              "label": conf_test["label"], "text": conf_test["text"], "pred": [labels[i] for i in conf_pred],
              "p_real": np.round(conf_p_real, 3)}).to_csv(f"{OUT}/{NAME}_confounder_test_predictions.csv", index=False)
with open(f"{OUT}/{NAME}_result.json", "w") as f:
    json.dump(result, f, indent=2)
log("finished")
'''
with open("worker.py", "w") as f:
    f.write(WORKER)

## 3. Train the queues in parallel

In [ ]:
KEEP = ("[", "Traceback", "Error", "error")  # progress lines and failures only


def run_queue(gpu, jobs):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "DEBUG": "1" if DEBUG else "0", "T0": str(T0),
           "TOKENIZERS_PARALLELISM": "false", "TRANSFORMERS_VERBOSITY": "error", "PYTHONWARNINGS": "ignore"}
    for name, cfg in jobs:
        log(f"GPU {gpu}: starting {name}")
        proc = subprocess.Popen(["python", "-u", "worker.py", name, json.dumps(cfg)], env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        tail = []
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            if line.startswith(KEEP):
                print(line, end="", flush=True)
        if proc.wait() != 0:
            log(f"GPU {gpu}: {name} FAILED — last output:\n" + "".join(tail))


threads = [threading.Thread(target=run_queue, args=(g, q)) for g, q in enumerate(QUEUES)]
for t in threads:
    t.start()
for t in threads:
    t.join()
log("all queues finished")

## 4. Results

In [ ]:
names = [n for q in QUEUES for n, _ in q if os.path.exists(f"{OUT}/{n}_result.json")]
results = {n: json.load(open(f"{OUT}/{n}_result.json")) for n in names}
rows = {}
for n in names:
    h, c = results[n]["humaid_test"], results[n]["confounder_test"]
    rows[n] = {"humaid_acc": h["accuracy"], "humaid_macro_f1": h["macro_f1"], "real_kept": c["real_kept"],
               "lookalike_rejected": c["lookalike_rejected"], "balanced": c["balanced"],
               "routine": c["rejected_routine"], "figurative": c["rejected_figurative"],
               "past_or_fiction": c["rejected_past_or_fiction"], "drill": c["rejected_drill"]}
summary = pd.DataFrame(rows).T
pd.set_option("display.width", 250)
print(summary.to_string(), flush=True)
summary.to_csv(f"{OUT}/pipeline_summary.csv")
with open(f"{OUT}/pipeline_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

In [ ]:
# Side-by-side predictions on the confounder test split
wide = None
for n in names:
    p = pd.read_csv(f"{OUT}/{n}_confounder_test_predictions.csv")
    p = p.rename(columns={"pred": n}).drop(columns=["p_real"])
    wide = p if wide is None else wide.merge(p[["id", n]], on="id")
if wide is not None:
    wide.to_csv(f"{OUT}/confounder_test_all_runs.csv", index=False)
log("done")